# TP1 â€“ Continual Learning on Seq-CIFAR-10
**I309 â€“ VisiÃ³n Artificial Avanzada Â· Universidad de San AndrÃ©s**

This notebook is the single entry point for all experiments.
It follows the four stages of the assignment:

1. Dataset preparation (Seq-CIFAR-10, replay buffer)
2. Pre-training with Supervised Contrastive Learning (SupCon)
3. Continual Learning methods: Naive, EWC, LwF, CoÂ²L
4. Comparison of results (Class-IL and Task-IL)


## 0. Setup

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # avoid OpenMP conflict on Windows

import sys, os

# __file__ doesn't exist in Jupyter â€” use the working directory instead.
# Launch Jupyter from the repo root (the folder containing tp1.ipynb) and
# this will resolve correctly.
REPO_ROOT = os.path.abspath('')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'REPO_ROOT: {REPO_ROOT}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

PyTorch  : 2.10.0+cpu
REPO_ROOT: /content
Device   : cpu


In [2]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from data.dataset import SeqCIFAR10, TASK_CLASSES, CLASS_NAMES
from data.buffer import ReplayBuffer
from models.backbone import Backbone
from losses.supcon import SupConLoss
from losses.distillation import AsymmetricDistillationLoss
from methods.naive import NaiveFineTuning
from methods.ewc import EWC
from methods.lwf import LwF
from methods.co2l import Co2L
from methods.trainer import ContinualTrainer
from methods.pretrain import pretrain_supcon, train_linear_probe, load_pretrained_backbone
from utils.metrics import evaluate_class_il, evaluate_task_il, compute_forgetting, MetricsTracker
from utils.visualization import (
    plot_accuracy_curve, plot_forgetting_curve,
    plot_comparison, plot_forgetting_heatmap,
    plot_embeddings, plot_loss_curve,
    plot_pretrain_loss, plot_embedding_stages, plot_probe_accuracy)

print('All imports OK')

ModuleNotFoundError: No module named 'data'

## Stage 1 â€“ Dataset Preparation

Split CIFAR-10 into 5 sequential tasks (2 classes each) and set up the replay buffer.

| Task | Classes               | Train images |
|------|-----------------------|--------------|
| 0    | airplane, automobile  | 10 000       |
| 1    | bird, cat             | 10 000       |
| 2    | deer, dog             | 10 000       |
| 3    | frog, horse           | 10 000       |
| 4    | ship, truck           | 10 000       |

In [ ]:
DATA_ROOT   = './data'   # torchvision downloads CIFAR-10 here on first run
N_TASKS     = 5
BATCH_SIZE  = 128
BUFFER_SIZE = 200        # replay buffer capacity across all past tasks

seq_cifar = SeqCIFAR10(data_root=DATA_ROOT, n_tasks=N_TASKS, batch_size=BATCH_SIZE)
buffer    = ReplayBuffer(capacity=BUFFER_SIZE)

print(f'Task splits (class indices): {TASK_CLASSES}')
print()
for t in range(N_TASKS):
    names = seq_cifar.get_class_names(t)
    n_train = seq_cifar.task_size(t, train=True)
    n_test  = seq_cifar.task_size(t, train=False)
    print(f'  Task {t}: {names[0]:12s} + {names[1]:12s}  |  train={n_train:5d}  test={n_test:4d}')

In [ ]:
# --- Verify class filtering is correct ---
print('Verifying class filtering...')
for t in range(N_TASKS):
    c0, c1 = seq_cifar.get_classes(t)
    loader = seq_cifar.get_task_loader(t, train=True)
    seen_classes = set()
    for _, labels in loader:
        seen_classes.update(labels.tolist())
    assert seen_classes == {c0, c1}, f'Task {t}: expected {{{c0},{c1}}}, got {seen_classes}'
    print(f'  Task {t} OK â€” classes {seen_classes} = {seq_cifar.get_class_names(t)}')

print()
# Verify a SupCon loader returns (B, 2, C, H, W)
supcon_loader = seq_cifar.get_task_loader(0, train=True, supcon=True)
imgs, labels = next(iter(supcon_loader))
assert imgs.ndim == 5 and imgs.shape[1] == 2 and imgs.shape[2:] == (3, 32, 32), \
    f'Expected (B,2,3,32,32) but got {imgs.shape}'
print(f'SupCon loader batch shape: {imgs.shape}  â†’  (B={imgs.shape[0]}, 2 views, C=3, H=32, W=32)')
print()
print('All checks passed.')

## Stage 2 â€“ Pre-training with Supervised Contrastive Learning

Train the backbone encoder on Task 0 using SupCon loss, then attach and train a linear classification head.

In [ ]:
# ── Backbone ──────────────────────────────────────────────────────────────────
backbone_pretrained = Backbone(num_classes=10).to(DEVICE)

# ── SupCon pre-training on Task 0 (airplane / automobile) ─────────────────────
from methods.pretrain import pretrain_supcon, load_pretrained_backbone

SUPCON_EPOCHS    = 200        # set lower (e.g. 20) for a quick smoke-test
SUPCON_LR        = 0.5
SUPCON_TEMP      = 0.07
CHECKPOINT_DIR   = "checkpoints"
CHECKPOINT_EVERY = 50         # save intermediate ckpts every 50 epochs (0 = off)

pretrain_history = pretrain_supcon(
    backbone        = backbone_pretrained,
    seq_cifar       = seq_cifar,
    task_id         = 0,
    n_epochs        = SUPCON_EPOCHS,
    lr              = SUPCON_LR,
    temperature     = SUPCON_TEMP,
    device          = DEVICE,
    checkpoint_dir  = CHECKPOINT_DIR,
    checkpoint_every= CHECKPOINT_EVERY,
    save_stages     = True,
)

print(f"Pre-training done.  Final loss: {pretrain_history['loss_history'][-1]:.4f}")
print(f"Checkpoint saved to: {pretrain_history['checkpoint_path']}")


In [ ]:
# ── SupCon loss curve ─────────────────────────────────────────────────────────
from utils.visualization import plot_pretrain_loss

plot_pretrain_loss(
    loss_history = pretrain_history["loss_history"],
    save_path    = "imgs/supcon_pretrain_loss.png",
    title        = "SupCon Pre-training Loss — Task 0 (airplane / automobile)",
)
print("Saved: imgs/supcon_pretrain_loss.png")


In [ ]:
# ── Embedding projections: init / mid / final ─────────────────────────────────
# Loads backbone snapshots saved during pre-training and runs t-SNE on
# Task 0 test features (airplane / automobile), producing a 1x3 side-by-side.
from utils.visualization import plot_embedding_stages

test_loader_viz = seq_cifar.get_task_loader(0, train=False)

plot_embedding_stages(
    stage_paths       = pretrain_history["stage_paths"],
    backbone_template = backbone_pretrained,
    loader            = test_loader_viz,
    device            = DEVICE,
    class_names       = CLASS_NAMES,
    save_path         = "imgs/supcon_embeddings_stages.png",
    n_samples         = 1000,
    method            = "tsne",
)
print("Saved: imgs/supcon_embeddings_stages.png")


In [ ]:
# ── Linear probe on Task 0 ────────────────────────────────────────────────────
# Encoder is frozen inside train_linear_probe(); the function verifies this
# every epoch and raises RuntimeError if the freeze is ever violated.

LINEAR_EPOCHS = 100
LINEAR_LR     = 1.0

probe_history = train_linear_probe(
    backbone       = backbone_pretrained,
    seq_cifar      = seq_cifar,
    task_id        = 0,
    n_epochs       = LINEAR_EPOCHS,
    lr             = LINEAR_LR,
    device         = DEVICE,
    checkpoint_dir = CHECKPOINT_DIR,
)

print(f"Encoder frozen after probe: {backbone_pretrained.is_encoder_frozen}")
print(f"Linear probe — Task 0 test accuracy : {probe_history['test_acc']:.3f}")
print(f"Checkpoint: {probe_history['checkpoint_path']}")


In [ ]:
# ── Linear probe results ──────────────────────────────────────────────────────
from utils.visualization import plot_probe_accuracy

plot_probe_accuracy(
    probe_history = probe_history,
    save_path     = "imgs/linear_probe_curves.png",
    title         = "Linear Probe — Task 0",
)
print(f"Test accuracy: {probe_history['test_acc']:.3f}")
print("Saved: imgs/linear_probe_curves.png")


In [ ]:
# ── Stage 3 setup ────────────────────────────────────────────────────────────
import copy, json, os

# Starting checkpoint: backbone trained with SupCon + linear probe on Task 0.
# Unfreeze the encoder so every method can fine-tune the full model.
backbone_pretrained.unfreeze_encoder()

def make_eval_fn(seq_cifar, device, tracker):
    """Return an eval_fn for ContinualTrainer.

    After each task t builds test loaders for tasks 0..t, runs both
    evaluation scenarios, and feeds results into the MetricsTracker.
    Returns the avg metrics dict that ContinualTrainer prints.
    """
    def eval_fn(task_id, method):
        test_loaders = [seq_cifar.get_task_loader(t, train=False)
                        for t in range(task_id + 1)]
        class_il = evaluate_class_il(method.backbone, test_loaders, device)
        task_il  = evaluate_task_il(method.backbone,  test_loaders, device)
        tracker.update(task_id, class_il, task_il)
        return {"class_il": class_il["avg_acc"], "task_il": task_il["avg_acc"]}
    return eval_fn

def save_tracker(tracker, name):
    """Persist a MetricsTracker's data to checkpoints/<name>_results.json."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    data = {
        "class_il": {
            "acc_matrix":      tracker._class_il,
            "avg_acc_curve":   tracker._avg_class_il,
            "forgetting":      tracker.forgetting("class_il"),
            "avg_forgetting":  tracker.avg_forgetting("class_il"),
        },
        "task_il": {
            "acc_matrix":      tracker._task_il,
            "avg_acc_curve":   tracker._avg_task_il,
            "forgetting":      tracker.forgetting("task_il"),
            "avg_forgetting":  tracker.avg_forgetting("task_il"),
        },
    }
    path = os.path.join(CHECKPOINT_DIR, f"{name}_results.json")
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Results saved to {path}")
    return path

print("Stage 3 setup done — encoder unfrozen:", not backbone_pretrained.is_encoder_frozen)


In [ ]:
# ── 3.1 Naive Fine-Tuning ────────────────────────────────────────────────────
# Lower-bound baseline: sequential cross-entropy fine-tuning, no forgetting
# mitigation.  Initialised from the SupCon-pretrained encoder + linear probe.
naive_tracker  = MetricsTracker(n_tasks=N_TASKS)

# Deep copy so other methods start from the same pretrained weights.
naive_backbone = copy.deepcopy(backbone_pretrained)

naive = NaiveFineTuning(naive_backbone, device=DEVICE, lr=0.01)
naive_trainer = ContinualTrainer(
    naive, seq_cifar,
    n_epochs = 50,
    eval_fn  = make_eval_fn(seq_cifar, DEVICE, naive_tracker),
)
naive_results = naive_trainer.train_all_tasks()

# ── Save results ──────────────────────────────────────────────────────────────
save_tracker(naive_tracker, "naive")

# ── Summary table ─────────────────────────────────────────────────────────────
print(naive_tracker)
naive_tracker.summary_df()


In [ ]:
# ── 3.2 EWC ──────────────────────────────────────────────────────────────────
# Penalty on parameter drift weighted by diagonal Fisher Information Matrix.
# Fisher is estimated from task-t data in end_task(); penalty accumulates
# across tasks (one term per past task).
ewc_tracker  = MetricsTracker(n_tasks=N_TASKS)
ewc_backbone = copy.deepcopy(backbone_pretrained)

ewc = EWC(ewc_backbone, device=DEVICE, ewc_lambda=400.0)
ewc_trainer = ContinualTrainer(
    ewc, seq_cifar,
    n_epochs = 50,
    eval_fn  = make_eval_fn(seq_cifar, DEVICE, ewc_tracker),
)
ewc_results = ewc_trainer.train_all_tasks()

save_tracker(ewc_tracker, "ewc")
print(ewc_tracker)
ewc_tracker.summary_df()


In [ ]:
# ── 3.3 LwF ──────────────────────────────────────────────────────────────────
# Before each task > 0 a frozen teacher snapshot is taken in begin_task().
# Training loss = CE(new task) + alpha * KL(teacher_soft_targets || student).
# T^2 re-scales distillation gradient to match CE gradient magnitude.
lwf_tracker  = MetricsTracker(n_tasks=N_TASKS)
lwf_backbone = copy.deepcopy(backbone_pretrained)

lwf = LwF(lwf_backbone, device=DEVICE, alpha=1.0, temperature=2.0)
lwf_trainer = ContinualTrainer(
    lwf, seq_cifar,
    n_epochs = 50,
    eval_fn  = make_eval_fn(seq_cifar, DEVICE, lwf_tracker),
)
lwf_results = lwf_trainer.train_all_tasks()

save_tracker(lwf_tracker, "lwf")
print(lwf_tracker)
lwf_tracker.summary_df()


In [ ]:
# ── 3.4 Co²L ──────────────────────────────────────────────────────────────────
# SupCon on joint batch (current task + replay) + CE on current task.
# buffer.update_from_loader() called in end_task() after each task.
# Distillation term is stubbed (v2); teacher snapshot is already taken.
co2l_tracker  = MetricsTracker(n_tasks=N_TASKS)
co2l_backbone = copy.deepcopy(backbone_pretrained)
co2l_buffer   = ReplayBuffer(capacity=BUFFER_SIZE)

co2l = Co2L(
    backbone      = co2l_backbone,
    device        = DEVICE,
    seq_cifar     = seq_cifar,
    replay_buffer = co2l_buffer,
    temperature   = 0.07,
    lr            = 0.5,
    n_buf_samples = 64,
)
co2l_trainer = ContinualTrainer(
    co2l, seq_cifar,
    n_epochs = 100,
    eval_fn  = make_eval_fn(seq_cifar, DEVICE, co2l_tracker),
)
co2l_results = co2l_trainer.train_all_tasks()

save_tracker(co2l_tracker, "co2l")
print(co2l_tracker)
co2l_tracker.summary_df()


## Stage 4 — Comparison of Results

Aggregates Class-IL and Task-IL accuracy from all four methods and produces:
- per-method accuracy-vs-tasks curves (`imgs/`)
- full per-task accuracy matrices
- a single summary table (printed + exported to `checkpoints/results_summary.csv` and `results_full.json`)


In [ ]:
# ── Stage 4 setup: aggregate trackers ────────────────────────────────────────────
import pandas as pd, json as _json, os, math, numpy as np

# Friendly task labels for x-axis ticks
TASK_NAMES   = ["T0\nplane/car", "T1\nbird/cat", "T2\ndeer/dog",
                "T3\nfrog/horse", "T4\nship/truck"]
FORG_NAMES   = TASK_NAMES[:-1]   # last task has no forgetting entry

def load_tracker_from_json(path, n_tasks=5):
    """Reconstruct a MetricsTracker from a JSON file saved by save_tracker()."""
    with open(path) as f:
        data = _json.load(f)
    t = MetricsTracker(n_tasks=n_tasks)
    for c_row, t_row, c_avg, t_avg in zip(
        data["class_il"]["acc_matrix"],
        data["task_il"]["acc_matrix"],
        data["class_il"]["avg_acc_curve"],
        data["task_il"]["avg_acc_curve"],
    ):
        t._class_il.append(c_row)
        t._task_il.append(t_row)
        t._avg_class_il.append(c_avg)
        t._avg_task_il.append(t_avg)
    return t

def get_tracker(name, n_tasks=N_TASKS):
    """Return live tracker if available, else load from checkpoint JSON."""
    live = globals().get(f"{name.lower().replace('\u00b2','2').replace(' ','_')}_tracker")
    if live is not None and len(live._avg_class_il) > 0:
        return live
    path = os.path.join(CHECKPOINT_DIR, f"{name.lower().replace('\u00b2','2')}_results.json")
    if os.path.exists(path):
        print(f"  Loading {name} results from {path}")
        return load_tracker_from_json(path, n_tasks)
    raise FileNotFoundError(
        f"No live tracker and no checkpoint at {path}.  "
        f"Run Stage 3 cell for {name} first."
    )

all_trackers = {}
for _name in ("Naive", "EWC", "LwF", "Co\u00b2L"):
    try:
        all_trackers[_name] = get_tracker(_name)
        print(f"  {_name}: {all_trackers[_name]}")
    except FileNotFoundError as e:
        print(f"  WARNING: {e}")

print(f"\nMethods loaded: {list(all_trackers.keys())}")
assert all_trackers, "No trackers loaded \u2014 run at least one Stage 3 method first."
os.makedirs("imgs", exist_ok=True)

# ── Individual accuracy curves ───────────────────────────────────────────────
plot_accuracy_curve(
    {m: t.avg_acc_curve("class_il") for m, t in all_trackers.items()},
    scenario="Class-IL",
    save_path="imgs/class_il_accuracy.png",
    task_names=TASK_NAMES,
)
plot_accuracy_curve(
    {m: t.avg_acc_curve("task_il") for m, t in all_trackers.items()},
    scenario="Task-IL",
    save_path="imgs/task_il_accuracy.png",
    task_names=TASK_NAMES,
    y_min=0.5,
)
plot_forgetting_curve(
    {m: t.forgetting("class_il") for m, t in all_trackers.items()},
    save_path="imgs/forgetting.png",
    task_names=FORG_NAMES,
)

# ── Combined 3-panel comparison figure ─────────────────────────────────────────
plot_comparison(
    all_trackers,
    save_path="imgs/comparison.png",
    task_names=TASK_NAMES,
    y_min_task_il=0.5,
)

# ── Forgetting heatmaps ────────────────────────────────────────────────────────
plot_forgetting_heatmap(
    all_trackers, scenario="class_il",
    save_path="imgs/heatmap_class_il.png",
    task_names=TASK_NAMES,
)
plot_forgetting_heatmap(
    all_trackers, scenario="task_il",
    save_path="imgs/heatmap_task_il.png",
    task_names=TASK_NAMES,
)

print("Saved:")
for p in ["imgs/class_il_accuracy.png", "imgs/task_il_accuracy.png",
          "imgs/forgetting.png", "imgs/comparison.png",
          "imgs/heatmap_class_il.png", "imgs/heatmap_task_il.png"]:
    print(f"  {p}")


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows_summary = []
for method_name, tracker in all_trackers.items():
    c_curve = tracker.avg_acc_curve("class_il")
    t_curve = tracker.avg_acc_curve("task_il")
    rows_summary.append({
        "Method":             method_name,
        "Class-IL final":     round(c_curve[-1], 4),
        "Task-IL final":      round(t_curve[-1],  4),
        "Avg Forgetting (C)": round(tracker.avg_forgetting("class_il"), 4),
        "Avg Forgetting (T)": round(tracker.avg_forgetting("task_il"),  4),
    })

df_summary = pd.DataFrame(rows_summary).set_index("Method")

print("\n" + "="*60)
print("  Final results — all methods")
print("="*60)
print(df_summary.to_string())
print("="*60 + "\n")
df_summary


In [ ]:
# ── Per-task accuracy breakdown ───────────────────────────────────────────────
# Accuracy on each individual task *after all 5 tasks are trained*.
# Reveals which past tasks are forgotten most by each method.
import math

for scenario in ("class_il", "task_il"):
    label = "Class-IL" if scenario == "class_il" else "Task-IL"
    rows_pt = []
    for method_name, tracker in all_trackers.items():
        mat = tracker.acc_matrix(scenario)       # (T, T) lower-triangular ndarray
        final_row = mat[-1]                      # accuracies after all tasks
        row = {"Method": method_name}
        for t, acc in enumerate(final_row):
            row[f"Task {t}"] = round(float(acc), 3) if not math.isnan(float(acc)) else float("nan")
        rows_pt.append(row)
    df_pt = pd.DataFrame(rows_pt).set_index("Method")
    print(f"\n{label} — Per-task accuracy after training all 5 tasks:")
    print(df_pt.to_string())

print()
df_pt   # display last table inline


In [ ]:
# ── Export results ────────────────────────────────────────────────────────────
import numpy as np
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# CSV export (one row per method, report-ready)
csv_path = os.path.join(CHECKPOINT_DIR, "results_summary.csv")
df_summary.to_csv(csv_path)
print(f"Saved: {csv_path}")

# Full JSON export (complete curves + matrices for all methods)
full_export = {}
for method_name, tracker in all_trackers.items():
    mat_c = tracker.acc_matrix("class_il")
    mat_t = tracker.acc_matrix("task_il")
    full_export[method_name] = {
        "class_il": {
            "acc_matrix":     [[None if np.isnan(v) else round(float(v), 4)
                                 for v in row] for row in mat_c],
            "avg_acc_curve":  [round(v, 4) for v in tracker.avg_acc_curve("class_il")],
            "forgetting":     [round(v, 4) for v in tracker.forgetting("class_il")],
            "avg_forgetting": round(tracker.avg_forgetting("class_il"), 4),
        },
        "task_il": {
            "acc_matrix":     [[None if np.isnan(v) else round(float(v), 4)
                                 for v in row] for row in mat_t],
            "avg_acc_curve":  [round(v, 4) for v in tracker.avg_acc_curve("task_il")],
            "forgetting":     [round(v, 4) for v in tracker.forgetting("task_il")],
            "avg_forgetting": round(tracker.avg_forgetting("task_il"), 4),
        },
    }

json_path = os.path.join(CHECKPOINT_DIR, "results_full.json")
with open(json_path, "w") as f:
    _json.dump(full_export, f, indent=2)
print(f"Saved: {json_path}")
print("\nExport complete.")
